# Autocomplete and Autocorrect Data Analytics
## 2. Autocomplete Implementation and Evaluation

This notebook covers:
- Implementing multiple autocomplete algorithms
- Training models on text data
- Evaluating performance metrics
- Comparing different approaches

## 1. Setup and Load Data

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time

# Import custom modules
from data_preprocessing import TextPreprocessor, DataLoader
from autocomplete import (
    TrieautocompleteEngine,
    NGramAutocomplete,
    FrequencybasedAutocomplete,
    AutocompleteEvaluator
)
from utils import VisualizationHelper, PerformanceMetrics

VisualizationHelper.set_style()
%matplotlib inline

print("Libraries loaded successfully!")

In [ ]:
# Load preprocessed data
data = pd.read_csv('../data/preprocessed_data.csv')

print(f"Loaded {len(data)} records")
print(f"\nFirst few records:")
data.head()

In [ ]:
# Load vocabulary
with open('../data/vocabulary.txt', 'r') as f:
    vocabulary = [line.strip() for line in f.readlines()]

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Sample words: {vocabulary[:20]}")

## 2. Build Autocomplete Algorithms

In [ ]:
# Prepare training data (all tokens from texts)
all_tokens = []
for tokens_str in data['tokens']:
    # Safely evaluate the string representation
    try:
        import ast
        tokens = ast.literal_eval(tokens_str)
        all_tokens.extend(tokens)
    except:
        pass

print(f"Total tokens for training: {len(all_tokens)}")
print(f"Sample tokens: {all_tokens[:20]}")

### 2.1 Trie-based Autocomplete

In [ ]:
# Train Trie-based autocomplete
print("Training Trie-based autocomplete...")
start_time = time.time()

trie_engine = TrieautocompleteEngine()
trie_engine.build_from_text(all_tokens)

trie_train_time = time.time() - start_time
print(f"Training time: {trie_train_time:.4f}s")

# Test predictions
test_prefix = "mach"
predictions = trie_engine.get_predictions(test_prefix, max_results=5)
print(f"\nPredictions for '{test_prefix}': {predictions}")

### 2.2 N-gram Autocomplete

In [ ]:
# Train N-gram autocomplete
print("Training N-gram autocomplete...")
start_time = time.time()

ngram_engine = NGramAutocomplete(n=3)
ngram_engine.train(all_tokens)

ngram_train_time = time.time() - start_time
print(f"Training time: {ngram_train_time:.4f}s")

# Test predictions
test_tokens = ['machine', 'learning']
predictions = ngram_engine.predict_next_word(test_tokens, top_k=5)
print(f"\nNext word predictions for {test_tokens}: {predictions}")

### 2.3 Frequency-based Autocomplete

In [ ]:
# Train frequency-based autocomplete
print("Training frequency-based autocomplete...")
start_time = time.time()

freq_engine = FrequencybasedAutocomplete()
freq_engine.train(all_tokens)

freq_train_time = time.time() - start_time
print(f"Training time: {freq_train_time:.4f}s")

# Test predictions
test_prefix = "mach"
predictions = freq_engine.predict(test_prefix, top_k=5)
print(f"\nPredictions for '{test_prefix}': {predictions}")

## 3. Performance Evaluation

In [ ]:
# Create test set
test_prefixes = ['mach', 'learn', 'data', 'nat', 'art', 'deep', 'net']
test_targets = ['machine', 'learning', 'data', 'natural', 'artificial', 'deep', 'network']

results = {}

# Test each model
models = {
    'Trie': trie_engine,
    'N-gram': ngram_engine,
    'Frequency': freq_engine
}

for model_name, model in models.items():
    print(f"\nTesting {model_name} model:")
    
    mrr_scores = []
    precision_scores = []
    query_times = []
    
    for prefix, target in zip(test_prefixes, test_targets):
        # Time the query
        start = time.time()
        predictions = model.predict(prefix, top_k=5) if hasattr(model, 'predict') else model.get_predictions(prefix, max_results=5)
        query_time = time.time() - start
        
        # Calculate metrics
        mrr = AutocompleteEvaluator.mean_reciprocal_rank(predictions, target)
        precision = AutocompleteEvaluator.precision_at_k(predictions, target)
        
        mrr_scores.append(mrr)
        precision_scores.append(precision)
        query_times.append(query_time)
    
    results[model_name] = {
        'mean_reciprocal_rank': np.mean(mrr_scores),
        'precision_at_5': np.mean(precision_scores),
        'avg_query_time': np.mean(query_times),
        'training_time': [trie_train_time, ngram_train_time, freq_train_time][["Trie", "N-gram", "Frequency"].index(model_name)]
    }
    
    print(f"  MRR: {results[model_name]['mean_reciprocal_rank']:.4f}")
    print(f"  Precision@5: {results[model_name]['precision_at_5']:.4f}")
    print(f"  Avg Query Time: {results[model_name]['avg_query_time']*1000:.4f}ms")

## 4. Results Comparison

In [ ]:
# Create results dataframe for easy viewing
results_df = pd.DataFrame(results).T
print("\nPerformance Comparison:")
print(results_df)

# Save results
results_df.to_csv('../results/autocomplete_comparison.csv')
print("\n✓ Results saved to '../results/autocomplete_comparison.csv'")

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# MRR Comparison
mrr_values = [results[model]['mean_reciprocal_rank'] for model in models.keys()]
axes[0].bar(models.keys(), mrr_values, color=['steelblue', 'darkgreen', 'coral'])
axes[0].set_ylabel('Score')
axes[0].set_title('Mean Reciprocal Rank')
axes[0].set_ylim(0, 1)

# Precision Comparison
precision_values = [results[model]['precision_at_5'] for model in models.keys()]
axes[1].bar(models.keys(), precision_values, color=['steelblue', 'darkgreen', 'coral'])
axes[1].set_ylabel('Score')
axes[1].set_title('Precision@5')
axes[1].set_ylim(0, 1)

# Query Time Comparison
time_values = [results[model]['avg_query_time']*1000 for model in models.keys()]
axes[2].bar(models.keys(), time_values, color=['steelblue', 'darkgreen', 'coral'])
axes[2].set_ylabel('Time (ms)')
axes[2].set_title('Average Query Time')

plt.tight_layout()
plt.savefig('../results/autocomplete_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure saved to '../results/autocomplete_comparison.png'")